In [2]:
import numpy as np

import pandas as pd

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler


import statsmodels.api as sm
from statsmodels.formula.api import ols

from build_lagged_features import (
    DATE,
    DEFAULT_LAGS,
    TARGET,
    build_lagged_frame,
    read_table,
)

In [4]:
'''
# Read data
df = pd.read_excel("data/chlorophyll_student_2015_2023.xlsx", sheet_name="daily_data")

# Order dates
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Build the dataset (with default lags)
df_lagged = build_lagged_frame(df, lag_spec=DEFAULT_LAGS)
'''

# Read data with lags
df = read_table("C:/Users/marga/Documents/tecnico/AAut/Projeto-Machine-Learning/data/chlorophyll_student_2015_2023.xlsx")
df_lagged = build_lagged_frame(df)


# Separate features (X) and target (y)
feature_cols = [c for c in df_lagged.columns if c not in [DATE, TARGET]]

X = df_lagged[feature_cols]
y = df_lagged[TARGET]

# Chronological Train/Test Split (Train < 2023, Test >= 2023)
train_mask = df_lagged[DATE] < "2023-01-01"

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]


In [5]:

# OLS Regression

model1 = LinearRegression(fit_intercept=True)

model1.fit(X_train, y_train)
y_pred = model1.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print({"RMSE": rmse, "MAE": mae, "R2": r2})




#

X = sm.add_constant(X_train)
model2 = sm.OLS(y_train, X).fit()
print(model2.summary())




'''
# 7. Train and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}

    if hasattr(model, "alpha_"):
        print(f"[{name}] Optimal Alpha Penalty: {model.alpha_:.4f}")

# 8. Print Results
comparison_df = pd.DataFrame(results).T
print("\n--- Model Performance Comparison (2023 Test Set) ---")
print(comparison_df.round(4))
'''



{'RMSE': np.float64(0.5471009284446109), 'MAE': 0.2243488116555645, 'R2': 0.5035694401864308}
                             OLS Regression Results                            
Dep. Variable:     chlorophyll_a_mg_m3   R-squared:                       0.758
Model:                             OLS   Adj. R-squared:                  0.747
Method:                  Least Squares   F-statistic:                     65.41
Date:                 Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                         14:02:44   Log-Likelihood:                -638.13
No. Observations:                 2908   AIC:                             1544.
Df Residuals:                     2774   BIC:                             2345.
Df Model:                          133                                         
Covariance Type:             nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
---------

'\n# 7. Train and Evaluate\nresults = {}\n\nfor name, model in models.items():\n    model.fit(X_train, y_train)\n    y_pred = model.predict(X_test)\n\n    rmse = np.sqrt(mean_squared_error(y_test, y_pred))\n    mae = mean_absolute_error(y_test, y_pred)\n    r2 = r2_score(y_test, y_pred)\n\n    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}\n\n    if hasattr(model, "alpha_"):\n        print(f"[{name}] Optimal Alpha Penalty: {model.alpha_:.4f}")\n\n# 8. Print Results\ncomparison_df = pd.DataFrame(results).T\nprint("\n--- Model Performance Comparison (2023 Test Set) ---")\nprint(comparison_df.round(4))\n'

In [ ]:
# Métricas para ver se o modelo é bom

# Para X_test

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print({"RMSE": rmse, "MAE": mae, "R2": r2})



# Para X_train

In [6]:
# 5. Scale Features (Crucial for Ridge and Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Define Cross-Validation and Models
tscv = TimeSeriesSplit(n_splits=5)
alphas = np.logspace(-3, 3, 100)  # Search range: 0.001 to 1000

models = {
    "OLS Linear Regression": LinearRegression(fit_intercept=True),
    "Ridge Regression (L2)": RidgeCV(alphas=alphas, cv=tscv),
    "Lasso Regression (L1)": LassoCV(
        alphas=alphas, cv=tscv, max_iter=10000, random_state=42
    ),
}

# 7. Train and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}

    if hasattr(model, "alpha_"):
        print(f"[{name}] Optimal Alpha Penalty: {model.alpha_:.4f}")

# 8. Print Results
comparison_df = pd.DataFrame(results).T
print("\n--- Model Performance Comparison (2023 Test Set) ---")
print(comparison_df.round(4))

[Ridge Regression (L2)] Optimal Alpha Penalty: 70.5480
[Lasso Regression (L1)] Optimal Alpha Penalty: 0.0093

--- Model Performance Comparison (2023 Test Set) ---
                         RMSE     MAE      R2
OLS Linear Regression  0.5471  0.2243  0.5036
Ridge Regression (L2)  0.5437  0.2228  0.5098
Lasso Regression (L1)  0.5442  0.2181  0.5088


In [1]:
from pathlib import Path
import build_lagged_features as blf   # works because the notebook is in the same folder as the .py file

def find_project_root(start=None):
    """Walk upward from the notebook's folder until a 'data/' folder is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data").is_dir():
            return folder
    raise FileNotFoundError(f"No 'data' folder found above {start}")

DATA_DIR = find_project_root() / "data"                        # the folder
DATA_FILE = DATA_DIR / "chlorophyll_student_2015_2023.xlsx"    # the input file
print(DATA_DIR)
print(DATA_FILE)

C:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\data
C:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\data\chlorophyll_student_2015_2023.xlsx


In [3]:
def make_lags(**overrides):
    """Copy of the default lags with changes.
    int N -> lags 0..N (1..N for the target); list -> used as is; None -> drop the variable."""
    spec = {k: list(v) for k, v in blf.DEFAULT_LAGS.items()}
    for col, val in overrides.items():
        if val is None:
            spec.pop(col, None)
        elif isinstance(val, int):
            spec[col] = list(range(1 if col == blf.TARGET else 0, val + 1))
        else:
            spec[col] = list(val)
    return spec

def build_and_save(raw, name, lag_spec=None, fmt="csv"):
    """Build the lagged frame and save it as <project>/data/<name>.<fmt>."""
    lagged = blf.build_lagged_frame(raw, lag_spec=lag_spec)
    path = DATA_DIR / f"{name}.{fmt}"
    if fmt == "xlsx":
        lagged.to_excel(path, sheet_name="lagged_data", index=False)
    else:
        lagged.to_csv(path, index=False)
    print(f"saved {lagged.shape} to {path}")
    return lagged

In [4]:
raw = blf.read_table(DATA_FILE)

# default lags
lagged = build_and_save(raw, "lagged_default")

# custom lags
my_lags = make_lags(
    chlorophyll_a_mg_m3=7,      # lags 1..7
    sst_c=[0, 1, 3, 7],         # specific lags
    turbidity_ntu=None,         # drop this variable
)
lagged_short = build_and_save(raw, "lagged_short", lag_spec=my_lags)

saved (3273, 135) to C:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\data\lagged_default.csv
saved (3273, 109) to C:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\data\lagged_short.csv
